<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/notebooks/01-processamento_pln.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

📑 Guia de Execução Estratégica
⚠️ IMPORTANTE: Sempre que o Runtime (Ambiente de Execução) for reiniciado, as Células 1 e 2 devem ser executadas obrigatoriamente para restabelecer os caminhos do Drive e reinstalar as bibliotecas.

🔄 Fluxo de Dependências:
Sessão Recém-Iniciada: Executar Célula 1 ➔ Célula 2.

Primeira vez no projeto: Executar Célula 1 ➔ Célula 2 ➔ Célula 3 (Carga).

Retomando Processamento: Se o banco já existe no Drive, pule a Célula 3 e vá direto para a Célula 4 e/ou 5 e/ou 6.

In [ ]:
# Célula 1: Montagem do Google Drive e Configuração de Caminhos
from google.colab import drive
import os

# 1. Montagem Segura: Só executa se ainda não estiver montado
if not os.path.exists('/content/drive/MyDrive'):
    print("📂 Montando Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive já está montado e acessível.")

# 2. Configuração Estrita de Caminhos
DRIVE_DIR = '/content/drive/MyDrive/mba-engsof-tcc/versao_final'
DB_FILE_NAME = 'data/base-dados.db'
DB_PATH = os.path.join(DRIVE_DIR, DB_FILE_NAME)

# Artefatos SQL
SCHEMA_SQL = os.path.join(DRIVE_DIR, 'sql/01-schema.sql')
SEED_SQL = os.path.join(DRIVE_DIR, 'sql/02-seed_data.sql')

# Pasta de Saída (Outputs)
EXPORT_PATH = os.path.join(DRIVE_DIR, 'outputs')
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📁 Pasta de exportação criada em: {EXPORT_PATH}")

print(f"📍 Banco de Dados: {DB_PATH}")

In [ ]:
# Célula 2: Instalação das bibliotecas e inicialização da estrutura (Schema)

# 1. Instalação Silenciosa
!pip install -q transformers torch pandas bertopic pysentimiento spacy
!python -m spacy download pt_core_news_lg -q

import sqlite3
import torch

# 2. Hardware Check para BERTimbau/BERTopic
device = 0 if torch.cuda.is_available() else -1

def inicializar_estrutura_db(db_path, schema_path):
    """Garante que a estrutura de tabelas esteja presente."""
    print(f"🛠️ Verificando integridade das tabelas...")

    # Se o arquivo de banco não existir, o SQLite o criará automaticamente
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        with open(schema_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())
        conn.commit()
        print("✅ Estrutura (Schema) validada com sucesso!")
    except Exception as e:
        print(f"❌ Erro ao processar Schema: {e}")
    finally:
        conn.close()

# 3. Execução
inicializar_estrutura_db(DB_PATH, SCHEMA_SQL)

print(f"\n🚀 Ambiente pronto (GPU: {'Ativa' if device == 0 else 'Inativa'}).")

In [ ]:
# Célula 3: Carga Inicial de Dados (Seed SQL)
def executar_carga_dados(db_path, seed_path):
    """Popula o banco apenas se a tabela 'verso' estiver vazia."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Verifica se já existem dados para evitar duplicidade no Drive
        cursor.execute("SELECT count(*) FROM verso")
        total_existente = cursor.fetchone()[0]

        if total_existente > 0:
            print(f"ℹ️ O banco já contém {total_existente} versos. Carga inicial ignorada.")
            return

        print("🌱 Semeando dados iniciais (02-seed_data.sql)... Isso pode levar alguns minutos.")
        with open(seed_path, 'r', encoding='utf-8') as f:
            cursor.executescript(f.read())

        conn.commit()
        print(f"✅ Carga de {seed_path} concluída com sucesso!")

    except sqlite3.OperationalError as e:
        print(f"⚠️ Erro operacional: {e}. Verifique se a Célula 2 foi executada.")
    except Exception as e:
        print(f"❌ Erro crítico na carga: {e}")
    finally:
        conn.close()

# Executa a carga (Somente se necessário)
executar_carga_dados(DB_PATH, SEED_SQL)

In [ ]:
# Célula 4: Limpeza por Assinatura Sintática e Extração de Métricas XAI
import spacy
import sqlite3
import pandas as pd
import numpy as np

# Carrega o modelo de português
try:
    nlp = spacy.load("pt_core_news_lg")
except:
    !python -m spacy download pt_core_news_lg
    nlp = spacy.load("pt_core_news_lg")

def processar_verso_com_xai(texto):
    if not texto or len(texto.strip()) < 3:
        return "RUIDO_CURTO", 0.0, 0.0, 0.0, 0.0

    doc = nlp(texto)
    total_tokens = len(doc)

    # Contagem de categorias gramaticais para métricas de explicabilidade
    counts = {'ADJ': 0, 'ADV': 0, 'PROPN': 0, 'VERB': 0, 'NUM': 0, 'NOUN': 0}
    for t in doc:
        if t.pos_ in counts:
            counts[t.pos_] += 1

    # --- 1. Cálculo de Indicadores de IA Explicável (XAI) ---

    # Score Emocional: Densidade de qualificadores (Adjetivos e Advérbios)
    score_emocional = (counts['ADJ'] + counts['ADV']) / total_tokens

    # Score Informativo: Densidade de dados brutos (Nomes Próprios e Números)
    score_informativo = (counts['PROPN'] + counts['NUM']) / total_tokens

    # Score de Dinâmica: Presença de Verbos (Ação/Processo)
    score_acao = counts['VERB'] / total_tokens

    # Entropia Gramatical Simplificada: Diversidade da estrutura (Riqueza vs. Repetição)
    present_tags = [v for v in counts.values() if v > 0]
    entropia_gramatical = -sum([(v/total_tokens) * np.log(v/total_tokens + 1e-9) for v in present_tags]) if present_tags else 0.0

    # --- 2. Lógica Dinâmica de Injeção de Contexto (Âncoras para Célula 5) ---

    tag = ""
    # Assinatura de Dados/Genealogia: Alta densidade de nomes próprios, baixa carga adjetiva
    if score_informativo > 0.40 and score_emocional < 0.05:
        tag = "ESTRUTURA_NOMINAL_DADOS: "

    # Assinatura Técnica/Normativa: Alta densidade de numerais (medidas, rituais)
    elif counts['NUM'] / total_tokens > 0.12:
        tag = "ESTRUTURA_TECNICA_NORMATIVA: "

    # Assinatura Narrativa Direta: Alta ação, mas sem carga emocional (seca)
    elif score_acao > 0.22 and score_emocional == 0:
        tag = "ESTRUTURA_NARRATIVA_DIRETA: "

    texto_limpo = tag + texto.strip()

    return texto_limpo, float(score_emocional), float(score_informativo), float(score_acao), float(entropia_gramatical)

# --- EXECUÇÃO E PERSISTÊNCIA ---

conn = sqlite3.connect(DB_PATH)
df_versos = pd.read_sql_query("SELECT id, texto FROM verso", conn)

print("🧠 Analisando assinaturas sintáticas e gerando scores XAI...")

# Processamento e expansão para novas colunas
resultados = df_versos['texto'].apply(processar_verso_com_xai)
df_versos[['texto_limpo', 'score_emocional', 'score_informativo', 'score_acao', 'entropia_gramatical']] = pd.DataFrame(resultados.tolist(), index=df_versos.index)

# Limpeza da tabela para nova persistência (sem DROPAR a estrutura)
cursor = conn.cursor()
cursor.execute("DELETE FROM verso_limpo")

# Persistência dos dados processados
df_versos[['id', 'texto_limpo', 'score_emocional', 'score_informativo', 'score_acao', 'entropia_gramatical']].rename(
    columns={'id': 'verso_id'}
).to_sql('verso_limpo', conn, if_exists='append', index=False)

conn.commit()
conn.close()

print(f"✅ Célula 4 concluída! {len(df_versos)} versos processados e persistidos em verso_limpo.")

In [ ]:
# Célula 5: Classificação por Eixos Existenciais com Governança Rígida XAI
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from transformers import pipeline
import sqlite3
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# 1. Configuração e Carga de Dados (Com nomes de colunas XAI atualizados)
conn = sqlite3.connect(DB_PATH)
query = """
    SELECT v.id as verso_id, v.texto, l.abreviacao, g.id as genero_id,
           vl.texto_limpo, vl.score_emocional, vl.score_informativo,
           vl.score_acao, vl.entropia_gramatical
    FROM verso v
    JOIN verso_limpo vl ON v.id = vl.verso_id
    JOIN livro l ON l.id = v.livro_id
    JOIN genero_literario g ON g.id = l.genero_id
"""
df_input = pd.read_sql_query(query, conn)

# Garantir que o texto para classificação seja o texto_limpo (com as âncoras da Célula 4)
docs_para_classificar = df_input['texto_limpo'].fillna('vazio').astype(str).tolist()

# 2. Inicialização do Modelo BERTimbau (Zero-Shot)
embedding_model = pipeline("feature-extraction", model="neuralmind/bert-base-portuguese-cased", device=0)

descricoes_eixos = [
    "Esgotamento e Alívio da Alma: Foca no abatimento espiritual, angústia mental, cansaço físico real e a fadiga do coração. Descreve o peso da opressão e o processo de restauração das energias.",
    "Temporalidade vs. Estabilidade Eterna: Foca na brevidade da existência, naquilo que é efêmero versus a solidez inabalável e eterna. Aborda a segurança de fundamentos permanentes.",
    "Vazio Existencial vs. Missão e Destino: Foca na crise de significado (vaidade) ou, na eleição, propósito de vida, chamado vocacional e planos futuros.",
    "Registros Narrativos, Genealogias e Dados: Textos informativos, listas de nomes próprios, sucessão de gerações, medidas técnicas, rituais, censos e ordens administrativas."
]

model_topic = BERTopic(
    embedding_model=embedding_model,
    zeroshot_topic_list=descricoes_eixos,
    zeroshot_min_similarity=0.1,
    calculate_probabilities=True,
    vectorizer_model=CountVectorizer(ngram_range=(1, 2))
)

print("🤖 Classificando via Zero-Shot e extraindo matriz de probabilidades...")
topics, probs_matrix = model_topic.fit_transform(docs_para_classificar)

# 3. Processamento de Decisões com Governança Rígida (Lockdown XAI)
rows_to_persist = []

print("⚖️ Aplicando Bloqueio Hard XAI e thresholds de precisão...")

for i, row in tqdm(df_input.iterrows(), total=len(df_input), desc="Processando Versículos", unit="v"):
    # Probabilidades do Modelo
    p_ex = probs_matrix[i][0] if probs_matrix.shape[1] > 0 else 0.0
    p_tr = probs_matrix[i][1] if probs_matrix.shape[1] > 1 else 0.0
    p_va = probs_matrix[i][2] if probs_matrix.shape[1] > 2 else 0.0
    p_na = probs_matrix[i][3] if probs_matrix.shape[1] > 3 else 0.0

    todas_probs = sorted([p_ex, p_tr, p_va, p_na], reverse=True)
    gap = todas_probs[0] - todas_probs[1]
    entropia = -sum([p * np.log(p + 1e-9) for p in [p_ex, p_tr, p_va, p_na]])

    existenciais = [p_ex, p_tr, p_va]
    best_idx = np.argmax(existenciais)
    best_score = existenciais[best_idx]

    t_limpo = str(row['texto_limpo'])
    s_emo = row['score_emocional']
    s_inf = row['score_informativo']
    s_ent = row['entropia_gramatical']

    # --- LÓGICA DE GOVERNANÇA RÍGIDA (LOCKDOWN) ---

    decisao_id = 3 # Default: Narrativo
    status = "Indefinido"

    # REGRA 1: Bloqueio Hard por Tags da Célula 4 (Prioridade Máxima)
    if "ESTRUTURA_NOMINAL_DADOS" in t_limpo or "ESTRUTURA_TECNICA_NORMATIVA" in t_limpo:
        decisao_id = 3
        status = "XAI: Bloqueio Estrutural (Hard)"

    elif "ESTRUTURA_NARRATIVA_DIRETA" in t_limpo and s_emo < 0.05:
        decisao_id = 3
        status = "XAI: Bloqueio Narrativo (Hard)"

    # REGRA 2: Filtro de Dados Brutos (XAI Overrule adicional por score)
    elif s_inf > 0.45 and s_emo < 0.06:
        decisao_id = 3
        status = "XAI: Filtro de Dados por Score"

    # REGRA 3: Validação Existencial (BERT + XAI)
    else:
        # Threshold de entropia mais rígido para evitar dispersão em textos informativos
        is_existencial = best_score > p_na and entropia < 1.10

        if not is_existencial:
            decisao_id = 3
            status = "Narrativo/Informativo"
        else:
            # Alta Confiança: Consenso entre Semântica (BERT) e Sintaxe (XAI)
            if gap > 0.15 or (best_score > 0.60 and s_emo > 0.15):
                decisao_id = best_idx
                status = "Alta Confiança (BERT+XAI)"
            elif best_score > 0.45:
                decisao_id = best_idx
                status = "Sensibilidade (Ambiguidade Controlada)"
            else:
                decisao_id = 3
                status = "Filtro de Ruído"

    # REGRA DE OURO: Foco Absoluto
    if entropia < 0.50 and best_score > 0.80 and s_emo > 0.10:
        decisao_id = best_idx
        status = "Foco Absoluto"

    rows_to_persist.append({
        'verso_id': row['verso_id'],
        'topico_id': int(decisao_id),
        'p_exaustao': float(p_ex),
        'p_transitoriedade': float(p_tr),
        'p_vazio': float(p_va),
        'p_narrativo': float(p_na),
        'similaridade_final': float(best_score if decisao_id != 3 else p_na),
        'margem_dominancia': float(best_score - p_na),
        'status_decisao': status,
        'entropia': float(entropia),
        'gap_confianca': float(gap)
    })

# 4. Persistência Final
print("💾 Persistindo resultados finais...")
df_final = pd.DataFrame(rows_to_persist)
cursor = conn.cursor()
cursor.execute("DELETE FROM verso_topico")
cursor.execute("DELETE FROM topico")

mapa_eixos = {0: "Exaustão vs. Refrigério", 1: "Transitoriedade vs. Solidez", 2: "Vazio vs. Propósito", 3: "Narrativo/Normativo"}
pd.DataFrame([{'id': k, 'antidoto_referencia': v} for k, v in mapa_eixos.items()]).to_sql('topico', conn, if_exists='append', index=False)
df_final.to_sql('verso_topico', conn, if_exists='append', index=False)

conn.commit()
conn.close()
print(f"✨ Processamento concluído! {len(df_final)} versículos classificados.")

In [ ]:
# Célula 6: Análise de Sentimento Contextual e Cruzamento Existencial
from pysentimiento import create_analyzer
import pandas as pd
import sqlite3
from tqdm.auto import tqdm

# 1. Inicializar o Analisador
print("🚀 Carregando modelo Transformer para Sentimento (PT-BR)...")
# O analisador 'sentiment' para 'pt' é baseado em BERTimbau, ideal para o TCC
analyzer = create_analyzer(task="sentiment", lang="pt")

# 2. Busca do texto original e dos tópicos
conn = sqlite3.connect(DB_PATH)
df_input = pd.read_sql_query("""
    SELECT v.id as verso_id, v.texto, vt.topico_id
    FROM verso v
    JOIN verso_topico vt ON v.id = vt.verso_id
""", conn)

textos = df_input['texto'].tolist()
verso_ids = df_input['verso_id'].tolist()

# 3. Execução da análise em lotes (Aproveitando a GPU se disponível)
print(f"📊 Analisando carga emocional de {len(textos)} versículos...")
sentimentos = []
batch_size = 64
mapa_num = {'POS': 1, 'NEU': 0, 'NEG': -1}

# O predict em lote é significativamente mais rápido no Colab
for i in tqdm(range(0, len(textos), batch_size)):
    lote = textos[i:i + batch_size]
    ids_lote = verso_ids[i:i + batch_size]
    preds_lote = analyzer.predict(lote)

    for idx, p in enumerate(preds_lote):
        # Capturamos as probabilidades brutas para análises de incerteza se necessário
        sentimentos.append({
            'verso_id': ids_lote[idx],
            'label': p.output,
            'sentimento_num': mapa_num.get(p.output, 0),
            'score_pos': p.probas.get('POS', 0),
            'score_neg': p.probas.get('NEG', 0),
            'score_neu': p.probas.get('NEU', 0)
        })

df_sent = pd.DataFrame(sentimentos)

# 4. Persistência dos Resultados
try:
    cursor = conn.cursor()
    # Limpamos para garantir que a nova classificação da Célula 5 seja a única presente
    cursor.execute("DELETE FROM verso_sentimento")

    # Inserimos os novos resultados (Integridade referencial com 'verso_id')
    df_sent.to_sql('verso_sentimento', conn, if_exists='append', index=False)
    conn.commit()
    print("\n✅ Célula 6 concluída! Sentimentos processados e salvos com sucesso.")

    # 5. RESULTADO FINAL: O DIAGNÓSTICO (PROBLEMA) VS. A CURA (ANTÍDOTO)
    print("\n📈 RESUMO EXECUTIVO: PROBLEMÁTICA (CRISE) VS. ANTÍDOTO (CURA)")

    res_final = pd.read_sql_query("""
        SELECT
            t.antidoto_referencia as Eixo_Filosofico,
            COUNT(*) as Total_Versos,
            SUM(CASE WHEN vs.sentimento_num = 1 THEN 1 ELSE 0 END) as Antidotos_Cura,
            SUM(CASE WHEN vs.sentimento_num = -1 THEN 1 ELSE 0 END) as Problematica_Crise,
            ROUND(AVG(vs.sentimento_num), 3) as Polaridade_Media
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        WHERE t.id != 3 -- Foco nos eixos Han, Bauman e Frankl
        GROUP BY t.antidoto_referencia
        ORDER BY Polaridade_Media DESC
    """, conn)

    # Exibe a tabela formatada no Colab
    display(res_final)

except Exception as e:
    print(f"❌ Erro na persistência: {e}")
finally:
    conn.close()